# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 01 — Spotify dataset preprocessing with PySpark

**Dataset:** [Spotify Dataset 1921–2020, 600k+ Tracks](https://www.kaggle.com/datasets/yamaerenay/spotify-dataset-19212020-600k-tracks/data)  
**Source:** Spotify Web API, republished by Yamac Eren Ay on Kaggle  
**Goal:** audit and clean the track-level data, study numeric correlations, and save a model-ready Parquet dataset for `02_model_creation.ipynb`.

This notebook is deliberately explanatory: each transformation is paired with its reason, and the final cells print observations calculated from the current data rather than relying on handwritten claims.

## 1. Reproducible project setup

The setup cell finds the repository root even when VS Code starts the notebook from the `notebooks` directory. It also selects the project-local 64-bit JDK before importing PySpark. This avoids accidentally using the machine's 32-bit Java runtime.

In [ ]:
from __future__ import annotations

import math
import os
import platform
import subprocess
import sys
from pathlib import Path


def find_project_root(*starts: Path) -> Path:
    """Find the project from either VS Code's cwd or this kernel's .venv."""
    visited = set()
    for start in starts:
        for candidate in (start, *start.parents):
            if candidate in visited:
                continue
            visited.add(candidate)
            if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
                return candidate
    raise FileNotFoundError(
        "Could not find the art_xharra project. Select the 'Python (art_xharra PySpark)' kernel, "
        "then rerun this cell."
    )


kernel_project_hint = Path(sys.executable).resolve().parents[2]
PROJECT_ROOT = find_project_root(Path.cwd().resolve(), kernel_project_hint)
os.chdir(PROJECT_ROOT)

jdk_roots = sorted((PROJECT_ROOT / ".tools" / "jdk17").glob("jdk-*"))
if not jdk_roots:
    raise RuntimeError("Project JDK not found. Run scripts/setup_environment.ps1 first.")
os.environ["JAVA_HOME"] = str(jdk_roots[-1])
os.environ["PATH"] = f"{jdk_roots[-1] / 'bin'}{os.pathsep}{os.environ['PATH']}"

# Native Windows Spark needs this local helper when creating Parquet directories.
project_hadoop_home = PROJECT_ROOT / ".tools" / "hadoop"
if not (project_hadoop_home / "bin" / "winutils.exe").exists():
    raise RuntimeError("Windows Hadoop helper not found. Run scripts/setup_environment.ps1 first.")
os.environ["HADOOP_HOME"] = str(project_hadoop_home)
os.environ["PATH"] = f"{project_hadoop_home / 'bin'}{os.pathsep}{os.environ['PATH']}"

# Force Spark workers to use the same virtual-environment interpreter as this kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version.split()[0]} ({platform.architecture()[0]})")
print(f"JAVA_HOME    : {os.environ.get('JAVA_HOME', 'not set')}")
print(f"HADOOP_HOME  : {os.environ.get('HADOOP_HOME', 'not set')}")

## 2. Obtain and locate the Kaggle data

The repository does not commit the large source files. If `tracks.csv` is absent, the next cell calls `scripts/download_data.py`, which uses Kaggle's official `kagglehub` Python library. Public datasets can normally be downloaded anonymously.

In [ ]:
DATASET_HANDLE = "yamaerenay/spotify-dataset-19212020-600k-tracks/versions/1"
RAW_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIRECTORY = PROJECT_ROOT / "data" / "processed"
TRACKS_PATH = RAW_DIRECTORY / "tracks.csv"
ARTISTS_PATH = RAW_DIRECTORY / "artists.csv"
OUTPUT_PATH = PROCESSED_DIRECTORY / "tracks_clean.parquet"
ARTIST_JOIN_PATH = PROCESSED_DIRECTORY / "track_artist_joined.parquet"
ENRICHED_TRACKS_PATH = PROCESSED_DIRECTORY / "tracks_enriched.parquet"
FIGURES_DIRECTORY = PROJECT_ROOT / "reports" / "figures"

if not TRACKS_PATH.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "download_data.py")],
        check=True,
    )

PROCESSED_DIRECTORY.mkdir(parents=True, exist_ok=True)
FIGURES_DIRECTORY.mkdir(parents=True, exist_ok=True)

assert TRACKS_PATH.exists(), f"Missing required file: {TRACKS_PATH}"
print(f"tracks.csv : {TRACKS_PATH.stat().st_size / 1024**2:,.1f} MiB")
print(f"artists.csv: {ARTISTS_PATH.stat().st_size / 1024**2:,.1f} MiB" if ARTISTS_PATH.exists() else "artists.csv: not downloaded (not required in phase 1)")

## 3. Start Spark and load with an explicit schema

Using an explicit schema is safer and faster than asking Spark to infer types from 600k+ rows. Invalid numeric text becomes null in permissive mode and is therefore visible in the quality audit. `_corrupt_record` captures structurally malformed CSV rows.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("art_xharra_spotify_preprocessing")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "12")
    .config("spark.sql.ansi.enabled", "false")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")
print(f"Spark master : {spark.sparkContext.master}")
print(f"Parallelism  : {spark.sparkContext.defaultParallelism}")

In [ ]:
TRACK_SCHEMA = T.StructType([
    T.StructField("id", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("popularity", T.IntegerType(), True),
    T.StructField("duration_ms", T.IntegerType(), True),
    T.StructField("explicit", T.IntegerType(), True),
    T.StructField("artists", T.StringType(), True),
    T.StructField("id_artists", T.StringType(), True),
    T.StructField("release_date", T.StringType(), True),
    T.StructField("danceability", T.DoubleType(), True),
    T.StructField("energy", T.DoubleType(), True),
    T.StructField("key", T.IntegerType(), True),
    T.StructField("loudness", T.DoubleType(), True),
    T.StructField("mode", T.IntegerType(), True),
    T.StructField("speechiness", T.DoubleType(), True),
    T.StructField("acousticness", T.DoubleType(), True),
    T.StructField("instrumentalness", T.DoubleType(), True),
    T.StructField("liveness", T.DoubleType(), True),
    T.StructField("valence", T.DoubleType(), True),
    T.StructField("tempo", T.DoubleType(), True),
    T.StructField("time_signature", T.IntegerType(), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

raw_tracks = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("encoding", "UTF-8")
    .option("quote", '"')
    .option("escape", '"')
    .schema(TRACK_SCHEMA)
    .csv(str(TRACKS_PATH))
    .persist(StorageLevel.MEMORY_AND_DISK)
)

raw_row_count = raw_tracks.count()
print(f"Raw shape: {raw_row_count:,} rows × {len(raw_tracks.columns) - 1} source columns")
raw_tracks.printSchema()
raw_tracks.select("id", "name", "artists", "release_date", "popularity").show(5, truncate=45)

## 4. Data-quality audit

The checks below distinguish four problems that are often accidentally mixed together:

1. SQL null values produced by absent or unparseable input.
2. Blank strings in textual fields.
3. IEEE `NaN` values in floating-point fields.
4. Present values that violate the documented Spotify feature domains.

A zero is not automatically missing: for example, `popularity = 0` and `instrumentalness = 0` are legitimate values.

In [ ]:
source_fields = [field for field in raw_tracks.schema.fields if field.name != "_corrupt_record"]
missing_expressions = []
nan_expressions = []

for field in source_fields:
    column = F.col(field.name)
    if isinstance(field.dataType, T.StringType):
        missing = column.isNull() | (F.length(F.trim(column)) == 0)
    else:
        missing = column.isNull()
    missing_expressions.append(
        F.sum(F.when(missing, 1).otherwise(0)).alias(f"{field.name}__missing")
    )
    if isinstance(field.dataType, (T.DoubleType, T.FloatType)):
        nan_expressions.append(
            F.sum(F.when(F.isnan(column), 1).otherwise(0)).alias(f"{field.name}__nan")
        )

missing_result = raw_tracks.agg(*(missing_expressions + nan_expressions)).first().asDict()
quality_rows = []
for field in source_fields:
    missing_count = int(missing_result[f"{field.name}__missing"] or 0)
    nan_count = int(missing_result.get(f"{field.name}__nan", 0) or 0)
    quality_rows.append({
        "column": field.name,
        "type": field.dataType.simpleString(),
        "missing_or_blank": missing_count,
        "nan": nan_count,
        "missing_pct": round(100 * missing_count / raw_row_count, 6),
    })

quality_pdf = pd.DataFrame(quality_rows).sort_values(
    ["missing_or_blank", "nan"], ascending=False
).reset_index(drop=True)
display(quality_pdf)

In [ ]:
# Validity rules reflect Spotify's documented score ranges and basic physical constraints.
UNIT_INTERVAL_COLUMNS = [
    "danceability", "energy", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence",
]

VALIDITY_RULES = {
    "popularity": F.col("popularity").between(0, 100),
    "duration_ms": F.col("duration_ms") > 0,
    "explicit": F.col("explicit").isin(0, 1),
    "key": F.col("key").between(-1, 11),
    "mode": F.col("mode").isin(0, 1),
    "loudness": (~F.isnan("loudness")) & (F.abs(F.col("loudness")) != float("inf")),
    "tempo": (~F.isnan("tempo")) & (F.col("tempo") > 0),
    # Spotify normally uses 3–7, but legacy API records in this dataset also contain 0–2.
    "time_signature": F.col("time_signature").between(0, 7),
}
for column_name in UNIT_INTERVAL_COLUMNS:
    VALIDITY_RULES[column_name] = (
        (~F.isnan(column_name)) & F.col(column_name).between(0.0, 1.0)
    )

invalid_expressions = [
    F.sum(
        F.when(F.col(column_name).isNotNull() & (~rule), 1).otherwise(0)
    ).alias(column_name)
    for column_name, rule in VALIDITY_RULES.items()
]
invalid_result = raw_tracks.agg(*invalid_expressions).first().asDict()
invalid_pdf = pd.DataFrame(
    [{"column": name, "invalid_values": int(count or 0)} for name, count in invalid_result.items()]
).sort_values("invalid_values", ascending=False).reset_index(drop=True)
display(invalid_pdf)

In [ ]:
def parsed_release_date(column_name: str):
    value = F.trim(F.col(column_name))
    return (
        F.when(value.rlike(r"^\d{4}-\d{2}-\d{2}$"), F.to_date(value, "yyyy-MM-dd"))
        .when(value.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(value, F.lit("-01")), "yyyy-MM-dd"))
        .when(value.rlike(r"^\d{4}$"), F.to_date(F.concat(value, F.lit("-01-01")), "yyyy-MM-dd"))
    )

date_value = F.trim(F.col("release_date"))
supported_date_pattern = date_value.rlike(r"^\d{4}(-\d{2}(-\d{2})?)?$")
date_audit = raw_tracks.agg(
    F.sum(F.when(F.col("release_date").isNull() | (F.length(date_value) == 0), 1).otherwise(0)).alias("missing"),
    F.sum(F.when(F.col("release_date").isNotNull() & (~supported_date_pattern), 1).otherwise(0)).alias("unsupported_format"),
    F.sum(F.when(supported_date_pattern & parsed_release_date("release_date").isNull(), 1).otherwise(0)).alias("invalid_calendar_date"),
).first().asDict()

corrupt_row_count = raw_tracks.filter(F.col("_corrupt_record").isNotNull()).count()
duplicate_id_groups = (
    raw_tracks
    .where(F.col("id").isNotNull())
    .groupBy("id")
    .count()
    .where(F.col("count") > 1)
    .persist(StorageLevel.MEMORY_AND_DISK)
)
duplicate_group_count = duplicate_id_groups.count()
duplicate_extra_rows = duplicate_id_groups.agg(
    F.coalesce(F.sum(F.col("count") - 1), F.lit(0)).alias("extra")
).first()["extra"]
exact_duplicate_rows = raw_row_count - raw_tracks.drop("_corrupt_record").dropDuplicates().count()
replacement_character_counts = raw_tracks.agg(*[
    F.sum(F.when(F.instr(F.col(column_name), "�") > 0, 1).otherwise(0)).alias(column_name)
    for column_name in ["name", "artists", "id_artists"]
]).first().asDict()

print(f"Corrupt CSV rows             : {corrupt_row_count:,}")
print(f"Exact duplicate rows         : {exact_duplicate_rows:,}")
print(f"Duplicated track-ID groups   : {duplicate_group_count:,}")
print(f"Extra rows from duplicate IDs: {duplicate_extra_rows:,}")
print(f"Release-date audit            : {date_audit}")
print(f"Unicode replacement characters: {replacement_character_counts}")

## 5. Cleaning decisions

The cleaning policy is conservative and reproducible:

- Trim textual fields and convert empty critical strings to null.
- Repair the three verified titles containing the destructive Unicode replacement character, keyed by Spotify track ID.
- Convert out-of-domain numeric values to null before imputation, so impossible values cannot enter the model silently.
- Remove rows missing `id`, `name`, `artists`, or the target candidate `popularity`; these values cannot be reconstructed responsibly.
- Parse year-only and year-month release dates as the first day of their known period. The original `release_date` text is retained, so no false precision is hidden.
- Deduplicate by Spotify track ID, preferring the row with the highest popularity and most recent parsed release date.
- Median-impute continuous predictors and mode-impute discrete predictors only when missing values actually exist. A `<feature>_was_missing` indicator is added before each imputation.

These choices prepare the predictors while keeping the target untouched.

In [ ]:
STRING_COLUMNS = ["id", "name", "artists", "id_artists", "release_date"]
CONTINUOUS_FEATURES = [
    "duration_ms", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness",
    "liveness", "valence", "tempo",
]
DISCRETE_FEATURES = ["explicit", "key", "mode", "time_signature"]
MODEL_FEATURES = CONTINUOUS_FEATURES + DISCRETE_FEATURES

cleaned = raw_tracks.drop("_corrupt_record")
for column_name in STRING_COLUMNS:
    cleaned = cleaned.withColumn(column_name, F.trim(F.col(column_name)))
cleaned = cleaned.replace("", None, subset=STRING_COLUMNS)

# U+FFFD destroys the original character, so apply only three externally verified, ID-keyed repairs.
KNOWN_NAME_REPAIRS = {
    "3xuLoZQr2Lg5qTeN2LNBT0": "Liefde in Die Reën",
    "6HrusMc7Gx5d1BGqGNTQlA": "Al Lê Die Berge Nog So Blou",
    "21icIn2eMV8nI0aKlgRDsv": "Un Roman D'Amitié",
}
for track_id, repaired_name in KNOWN_NAME_REPAIRS.items():
    cleaned = cleaned.withColumn(
        "name", F.when(F.col("id") == track_id, F.lit(repaired_name)).otherwise(F.col("name"))
    )

# Invalid present values become null; this makes the later repair explicit and auditable.
for column_name, rule in VALIDITY_RULES.items():
    data_type = cleaned.schema[column_name].dataType
    cleaned = cleaned.withColumn(
        column_name,
        F.when(F.col(column_name).isNull() | (~rule), F.lit(None).cast(data_type))
        .otherwise(F.col(column_name)),
    )

cleaned = (
    cleaned
    .withColumn("release_date_parsed", parsed_release_date("release_date"))
    .withColumn("release_year", F.year("release_date_parsed"))
    .dropna(subset=["id", "name", "artists", "popularity"])
)

deduplication_window = Window.partitionBy("id").orderBy(
    F.col("popularity").desc_nulls_last(),
    F.col("release_date_parsed").desc_nulls_last(),
)
cleaned = (
    cleaned
    .withColumn("_dedupe_rank", F.row_number().over(deduplication_window))
    .where(F.col("_dedupe_rank") == 1)
    .drop("_dedupe_rank")
)

missing_predictor_counts = cleaned.agg(*[
    F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
    for column_name in MODEL_FEATURES
]).first().asDict()
features_requiring_repair = [
    name for name, count in missing_predictor_counts.items() if int(count or 0) > 0
]

for column_name in features_requiring_repair:
    cleaned = cleaned.withColumn(
        f"{column_name}_was_missing", F.col(column_name).isNull().cast("int")
    )

continuous_fill_values = {}
for column_name in CONTINUOUS_FEATURES:
    if column_name in features_requiring_repair:
        quantiles = cleaned.approxQuantile(column_name, [0.5], 0.001)
        if quantiles:
            continuous_fill_values[column_name] = quantiles[0]
if continuous_fill_values:
    cleaned = cleaned.fillna(continuous_fill_values)

discrete_fill_values = {}
for column_name in DISCRETE_FEATURES:
    if column_name in features_requiring_repair:
        mode_row = (
            cleaned.where(F.col(column_name).isNotNull())
            .groupBy(column_name).count()
            .orderBy(F.desc("count"), F.asc(column_name))
            .first()
        )
        if mode_row is not None:
            discrete_fill_values[column_name] = mode_row[column_name]
if discrete_fill_values:
    cleaned = cleaned.fillna(discrete_fill_values)

cleaned = cleaned.persist(StorageLevel.MEMORY_AND_DISK)
clean_row_count = cleaned.count()
print(f"Rows before cleaning : {raw_row_count:,}")
print(f"Rows after cleaning  : {clean_row_count:,}")
print(f"Rows removed         : {raw_row_count - clean_row_count:,}")
print(f"Predictors repaired  : {features_requiring_repair or 'none'}")
print(f"Continuous medians   : {continuous_fill_values or 'not needed'}")
print(f"Discrete modes       : {discrete_fill_values or 'not needed'}")

In [ ]:
# Post-cleaning assertions turn silent data drift into a visible notebook failure.
remaining_required_nulls = cleaned.agg(*[
    F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(column_name)
    for column_name in ["id", "name", "artists", "popularity", *MODEL_FEATURES]
]).first().asDict()
remaining_duplicate_ids = (
    cleaned.groupBy("id").count().where(F.col("count") > 1).limit(1).count()
)
remaining_damaged_names = cleaned.where(F.instr(F.col("name"), "�") > 0).count()

assert all(int(value or 0) == 0 for value in remaining_required_nulls.values()), remaining_required_nulls
assert remaining_duplicate_ids == 0, "Duplicate track IDs remain after cleaning."
assert remaining_damaged_names == 0, "A track name still contains a Unicode replacement character."
assert clean_row_count > 0, "Cleaning removed every row."

print("Post-cleaning validation passed.")
print(f"Required-field null counts: {remaining_required_nulls}")
print(f"Remaining duplicate IDs  : {remaining_duplicate_ids}")
print(f"Remaining damaged names : {remaining_damaged_names}")

## 6. Correlation matrix

The Pearson matrix uses the cleaned numeric columns. Encoded categories (`explicit`, `key`, `mode`, and `time_signature`) are included for completeness, but their Pearson coefficients must be interpreted cautiously because their numeric codes are not continuous measurements. Correlation describes linear association, not causation.

In [ ]:
CORRELATION_COLUMNS = [
    "popularity", "duration_ms", "explicit", "danceability", "energy",
    "key", "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
    "time_signature", "release_year",
]

correlation_input = cleaned.select(*CORRELATION_COLUMNS).na.drop()
correlation_row_count = correlation_input.count()
vectorized = VectorAssembler(
    inputCols=CORRELATION_COLUMNS, outputCol="features", handleInvalid="skip"
).transform(correlation_input).select("features")
correlation_matrix = Correlation.corr(vectorized, "features", "pearson").first()[0]
corr_pdf = pd.DataFrame(
    correlation_matrix.toArray(),
    index=CORRELATION_COLUMNS,
    columns=CORRELATION_COLUMNS,
)

print(f"Rows used for correlations: {correlation_row_count:,}")
display(corr_pdf.round(3))

In [ ]:
sns.set_theme(style="white", context="notebook")
figure, axis = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_pdf,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.35,
    cbar_kws={"label": "Pearson correlation", "shrink": 0.8},
    ax=axis,
)
axis.set_title("Spotify track numeric-feature correlation matrix", pad=16, weight="bold")
figure.tight_layout()
correlation_figure_path = FIGURES_DIRECTORY / "correlation_matrix.png"
figure.savefig(correlation_figure_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved figure: {correlation_figure_path.relative_to(PROJECT_ROOT)}")

In [ ]:
correlation_pairs = []
for left_index, left_name in enumerate(CORRELATION_COLUMNS):
    for right_index in range(left_index + 1, len(CORRELATION_COLUMNS)):
        right_name = CORRELATION_COLUMNS[right_index]
        value = float(corr_pdf.iloc[left_index, right_index])
        if not math.isnan(value):
            correlation_pairs.append({
                "feature_1": left_name,
                "feature_2": right_name,
                "correlation": value,
                "absolute_correlation": abs(value),
            })

strongest_pairs_pdf = (
    pd.DataFrame(correlation_pairs)
    .sort_values("absolute_correlation", ascending=False)
    .reset_index(drop=True)
)
popularity_correlations_pdf = (
    corr_pdf["popularity"]
    .drop("popularity")
    .rename("correlation_with_popularity")
    .to_frame()
    .assign(absolute_correlation=lambda frame: frame["correlation_with_popularity"].abs())
    .sort_values("absolute_correlation", ascending=False)
)

print("Strongest numeric relationships (excluding the diagonal):")
display(strongest_pairs_pdf.head(15).round(4))
print("Correlations with popularity (candidate target for phase 2):")
display(popularity_correlations_pdf.round(4))

## 7. Save the cleaned dataset

Parquet keeps the Spark schema, compresses the data, and is much faster to reload than CSV. The output is a directory containing partition files, which is the normal Spark format.

In [ ]:
(
    cleaned
    .repartition(8)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .parquet(str(OUTPUT_PATH))
)

verification_df = spark.read.parquet(str(OUTPUT_PATH))
written_row_count = verification_df.count()
assert written_row_count == clean_row_count
print(f"Saved {written_row_count:,} rows to {OUTPUT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved schema has {len(verification_df.columns)} columns.")

## 8. Relate tracks to artists and save joined tables

`tracks.csv` stores artist IDs as a list-like string, while `artists.csv` contains one row per artist. The next cells normalize that many-to-many relationship into a **long table with one row per track–artist pair**, then create a **one-row-per-track enriched table** with aggregate artist metrics for modeling.

The join stays left-sided from tracks to artists: a track relationship is never discarded merely because artist metadata is unavailable. Match coverage is measured explicitly.

In [ ]:
ARTIST_SCHEMA = T.StructType([
    T.StructField("id", T.StringType(), True),
    T.StructField("followers", T.DoubleType(), True),
    T.StructField("genres", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("popularity", T.IntegerType(), True),
    T.StructField("_corrupt_record", T.StringType(), True),
])

raw_artists = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .option("encoding", "UTF-8")
    .option("quote", '"')
    .option("escape", '"')
    .schema(ARTIST_SCHEMA)
    .csv(str(ARTISTS_PATH))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
artist_row_count = raw_artists.count()

artist_quality = raw_artists.agg(
    F.sum(F.when(F.col("id").isNull() | (F.length(F.trim("id")) == 0), 1).otherwise(0)).alias("missing_id"),
    F.sum(F.when(F.col("name").isNull() | (F.length(F.trim("name")) == 0), 1).otherwise(0)).alias("missing_name"),
    F.sum(F.when(F.col("followers").isNull() | F.isnan("followers"), 1).otherwise(0)).alias("missing_followers"),
    F.sum(F.when(F.col("popularity").isNull(), 1).otherwise(0)).alias("missing_popularity"),
    F.sum(F.when(F.col("_corrupt_record").isNotNull(), 1).otherwise(0)).alias("corrupt_rows"),
).first().asDict()

artist_id_duplicate_groups = (
    raw_artists.where(F.col("id").isNotNull())
    .groupBy("id").count().where(F.col("count") > 1).count()
)

clean_artists = raw_artists.drop("_corrupt_record")
for column_name in ["id", "genres", "name"]:
    clean_artists = clean_artists.withColumn(column_name, F.trim(F.col(column_name)))
clean_artists = clean_artists.replace("", None, subset=["id", "genres", "name"])
clean_artists = (
    clean_artists
    .withColumn(
        "followers",
        F.when(
            F.col("followers").isNotNull() & (~F.isnan("followers")) & (F.col("followers") >= 0),
            F.col("followers"),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "popularity",
        F.when(F.col("popularity").between(0, 100), F.col("popularity"))
        .otherwise(F.lit(None).cast("int")),
    )
    .dropna(subset=["id"])
)
artist_deduplication_window = Window.partitionBy("id").orderBy(
    F.col("followers").desc_nulls_last(), F.col("popularity").desc_nulls_last()
)
artist_dimension = (
    clean_artists
    .withColumn("_rank", F.row_number().over(artist_deduplication_window))
    .where(F.col("_rank") == 1)
    .select(
        F.col("id").alias("artist_id"),
        F.col("name").alias("artist_name"),
        F.col("followers").alias("artist_followers"),
        F.col("popularity").alias("artist_popularity"),
        F.coalesce(F.col("genres"), F.lit("[]")).alias("artist_genres"),
        F.lit(1).alias("_artist_match"),
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
artist_dimension_count = artist_dimension.count()

print(f"Artist source rows       : {artist_row_count:,}")
print(f"Artist dimension rows    : {artist_dimension_count:,}")
print(f"Duplicate artist-ID groups: {artist_id_duplicate_groups:,}")
print(f"Artist quality audit     : {artist_quality}")

In [ ]:
TRACK_COLUMN_RENAMES = {
    "id": "track_id",
    "name": "track_name",
    "popularity": "track_popularity",
    "artists": "track_artists_raw",
    "id_artists": "artist_ids_raw",
}
track_join_source = cleaned.select(*[
    F.col(column_name).alias(TRACK_COLUMN_RENAMES.get(column_name, column_name))
    for column_name in cleaned.columns
])

# Artist IDs are Spotify's 22-character alphanumeric IDs, so removing list punctuation is safe.
track_join_source = track_join_source.withColumn(
    "_artist_ids_array",
    F.when(
        F.trim(F.col("artist_ids_raw")) == "[]",
        F.array().cast("array<string>"),
    ).otherwise(
        F.split(F.regexp_replace(F.col("artist_ids_raw"), r"[\[\]']", ""), r",\s*")
    ),
)
tracks_without_artist_ids = track_join_source.where(F.size("_artist_ids_array") == 0).count()

relationship_source = (
    track_join_source
    .select("*", F.posexplode("_artist_ids_array").alias("artist_position", "artist_id"))
    .drop("_artist_ids_array")
    .where(F.col("artist_id").isNotNull() & (F.length(F.trim("artist_id")) > 0))
)
relationship_source_count = relationship_source.count()
malformed_artist_ids = relationship_source.where(
    ~F.col("artist_id").rlike(r"^[A-Za-z0-9]{22}$")
).count()
track_artist_relationships = (
    relationship_source
    .dropDuplicates(["track_id", "artist_id"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
relationship_count = track_artist_relationships.count()
duplicate_relationship_rows = relationship_source_count - relationship_count
bridge_track_count = track_artist_relationships.select("track_id").distinct().count()

joined_artist_tracks = (
    track_artist_relationships
    .join(artist_dimension, on="artist_id", how="left")
    .withColumn("artist_metadata_found", F.col("_artist_match").isNotNull())
    .drop("_artist_match")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
joined_relationship_count = joined_artist_tracks.count()
matched_relationship_count = joined_artist_tracks.where("artist_metadata_found").count()
unmatched_relationship_count = joined_relationship_count - matched_relationship_count
artist_match_pct = 100 * matched_relationship_count / joined_relationship_count
matched_track_ids = (
    joined_artist_tracks.where("artist_metadata_found").select("track_id").distinct()
)
tracks_without_artist_metadata = (
    cleaned.select(F.col("id").alias("track_id"))
    .join(matched_track_ids, on="track_id", how="left_anti")
    .count()
)

assert joined_relationship_count == relationship_count
assert bridge_track_count + tracks_without_artist_ids == clean_row_count
print(f"Track–artist relationships : {relationship_count:,}")
print(f"Tracks represented in bridge: {bridge_track_count:,} / {clean_row_count:,}")
print(f"Tracks without artist IDs   : {tracks_without_artist_ids:,}")
print(f"Malformed artist IDs        : {malformed_artist_ids:,}")
print(f"Duplicate relationship rows : {duplicate_relationship_rows:,}")
print(f"Matched relationships       : {matched_relationship_count:,} ({artist_match_pct:.3f}%)")
print(f"Unmatched relationships     : {unmatched_relationship_count:,}")
print(f"Tracks without matched metadata: {tracks_without_artist_metadata:,}")
joined_artist_tracks.select(
    "track_id", "track_name", "artist_position", "artist_id",
    "artist_name", "artist_followers", "artist_popularity", "artist_genres",
).show(10, truncate=35)

In [ ]:
(
    joined_artist_tracks.repartition(8)
    .write.mode("overwrite").option("compression", "snappy")
    .parquet(str(ARTIST_JOIN_PATH))
)

artist_features_by_track = joined_artist_tracks.groupBy("track_id").agg(
    F.countDistinct("artist_id").alias("artist_count"),
    F.sum(F.col("artist_metadata_found").cast("int")).alias("matched_artist_count"),
    F.avg("artist_followers").alias("artist_followers_mean"),
    F.max("artist_followers").alias("artist_followers_max"),
    F.avg("artist_popularity").alias("artist_popularity_mean"),
    F.max("artist_popularity").alias("artist_popularity_max"),
    F.array_sort(F.collect_set("artist_name")).alias("artist_names"),
    F.array_sort(F.collect_set("artist_genres")).alias("artist_genre_lists_raw"),
)
tracks_enriched = (
    cleaned
    .join(artist_features_by_track, cleaned.id == artist_features_by_track.track_id, "left")
    .drop(artist_features_by_track.track_id)
    .fillna({"artist_count": 0, "matched_artist_count": 0})
    .persist(StorageLevel.MEMORY_AND_DISK)
)
enriched_track_count = tracks_enriched.count()
assert enriched_track_count == clean_row_count

(
    tracks_enriched.repartition(8)
    .write.mode("overwrite").option("compression", "snappy")
    .parquet(str(ENRICHED_TRACKS_PATH))
)

saved_join_count = spark.read.parquet(str(ARTIST_JOIN_PATH)).count()
saved_enriched_count = spark.read.parquet(str(ENRICHED_TRACKS_PATH)).count()
assert saved_join_count == relationship_count
assert saved_enriched_count == clean_row_count

print(f"Saved long joined table : {ARTIST_JOIN_PATH.relative_to(PROJECT_ROOT)} ({saved_join_count:,} rows)")
print(f"Saved enriched tracks   : {ENRICHED_TRACKS_PATH.relative_to(PROJECT_ROOT)} ({saved_enriched_count:,} rows)")
print(f"Enriched track columns  : {len(tracks_enriched.columns)}")

## 9. Reproducible observations and hand-off

This cell summarizes the measurements produced above. Because it derives its text from computed variables, rerunning the notebook after a data update also refreshes the observations.

In [ ]:
columns_with_missing = quality_pdf.loc[quality_pdf["missing_or_blank"] > 0]
columns_with_nan = quality_pdf.loc[quality_pdf["nan"] > 0]
columns_with_invalid = invalid_pdf.loc[invalid_pdf["invalid_values"] > 0]
top_pair = strongest_pairs_pdf.iloc[0]
top_popularity = popularity_correlations_pdf.iloc[0]

print("OBSERVATIONS")
print("============")
print(f"1. The source contains {raw_row_count:,} track rows and {len(source_fields)} columns.")
if columns_with_missing.empty:
    print("2. No SQL nulls or blank strings were found in the source columns.")
else:
    details = ", ".join(
        f"{row.column}={int(row.missing_or_blank):,}" for row in columns_with_missing.itertuples()
    )
    print(f"2. Missing/blank values were found in: {details}.")
if columns_with_nan.empty:
    print("3. No IEEE NaN values were found in floating-point columns.")
else:
    details = ", ".join(f"{row.column}={int(row.nan):,}" for row in columns_with_nan.itertuples())
    print(f"3. NaN values were found in: {details}.")
if columns_with_invalid.empty:
    print("4. All present numeric values passed the declared domain checks.")
else:
    details = ", ".join(
        f"{row.column}={int(row.invalid_values):,}" for row in columns_with_invalid.itertuples()
    )
    print(f"4. Out-of-domain numeric values were found in: {details}.")
repaired_text_count = sum(int(value or 0) for value in replacement_character_counts.values())
print(f"5. {repaired_text_count:,} damaged track names were repaired with the verified ID-keyed title map.")
print(
    f"6. Structural audit: {corrupt_row_count:,} corrupt rows, {exact_duplicate_rows:,} exact duplicate rows, "
    f"and {duplicate_extra_rows:,} extra rows sharing a track ID."
)
print(
    f"7. Cleaning retained {clean_row_count:,} rows ({100 * clean_row_count / raw_row_count:.3f}% of the source)."
)
release_bounds = cleaned.agg(F.min("release_year").alias("min"), F.max("release_year").alias("max")).first()
print(f"8. Parsed release years span {release_bounds['min']}–{release_bounds['max']}, wider than the dataset title suggests.")
print(
    f"9. The strongest numeric pair is {top_pair.feature_1} ↔ {top_pair.feature_2} "
    f"(r={top_pair.correlation:.3f})."
)
print(
    f"10. The largest absolute univariate correlation with popularity is "
    f"{popularity_correlations_pdf.index[0]} (r={top_popularity.correlation_with_popularity:.3f}). "
    "This is descriptive evidence, not a model-performance estimate."
)
print(
    f"11. The artist source has {artist_row_count:,} rows; deduplication produced "
    f"{artist_dimension_count:,} unique artist IDs."
)
print(
    f"12. The bridge contains {relationship_count:,} track–artist relationships with "
    f"{artist_match_pct:.3f}% artist-metadata coverage ({unmatched_relationship_count:,} unmatched)."
)
print(
    f"13. Saved a long joined table ({saved_join_count:,} rows) and a one-row-per-track "
    f"enriched table ({saved_enriched_count:,} rows) for phase 2."
)
print(
    "14. Phase 2 should use time-aware validation and compare models with and without release_year, "
    "because its strong popularity association may reflect a collection-time cohort effect."
)

In [ ]:
# Compact statistical profile for reviewers; percentiles are approximate for scalability.
profile_columns = [
    "popularity", "duration_ms", "danceability", "energy", "loudness",
    "speechiness", "acousticness", "instrumentalness", "liveness",
    "valence", "tempo", "release_year",
]
cleaned.select(*profile_columns).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(20, truncate=False)